# Name That LLM · 08 公开扩展数据 + 加权多视图 SVM

目标为提高 18 类平均召回率；离线结果不能保证 Kaggle 分数。CPU 即可，无需 GPU、预训练权重或联网安装依赖。

**必须添加两个输入**：比赛 Name That LLM；[Name That LLM Open Pool，版本 18](https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool/versions/18)。使用该版本原始 parquet 的 SHA-256 核对输入，清洗后每类固定抽取 3,000 条。关闭 Internet，再 Save Version → Save & Run All；完成后提交 `submission.parquet`。

方法：字符、词、词形和功能词四组 TF-IDF；LinearSVC(C=0.5)；官方样本基础权重 12、外部样本 1，再按有效样本权重平衡类别。避免对隐藏测试进行类别配额分配或假设每个提示词有完整 18 类。

本项目实现代码；公开思路参考 [SSanjay0614](https://github.com/SSanjay0614/Name-That-LLM-Kaggle)、[stewari1210](https://github.com/stewari1210/name-that-llm)、[richardharnisch](https://github.com/richardharnisch/name-that-llm)。外部数据归 crtproduction 及其 sources.csv 列明的原作者，CC BY-SA 4.0；本 Notebook 不分发原始数据。外部清洗曾参考全部官方训练集排版统计，离线验证存在这一局限；不使用发布者的 match_weight/style_score。

默认仅训练固定配置并输出。设置 `NAME_THAT_LLM_VALIDATE=1` 可复现开发三折和本轮留出提示词评估；历史试验用过全体官方数据，因此留出不能称为历史上未用过的独立测试集。


In [ ]:
from pathlib import Path
import os, json, hashlib, time
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from threadpoolctl import threadpool_limits

STARTED=time.monotonic()
VALIDATE=os.environ.get('NAME_THAT_LLM_VALIDATE','0')=='1'
OUTPUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
DATA_CANDIDATES=[Path(os.environ['NAME_THAT_LLM_DATA'])] if 'NAME_THAT_LLM_DATA' in os.environ else [
    Path('/kaggle/input/competitions/name-that-llm'),Path('/kaggle/input/name-that-llm'),
    Path('data'),Path.cwd().parent/'data',
]
valid_dirs=[p for p in DATA_CANDIDATES if (p/'train.csv').is_file() and (p/'test.csv').is_file()]
if not valid_dirs:
    raise FileNotFoundError('Add competition input Name That LLM; local runs use --data-dir.')
DATA_DIR=valid_dirs[0]
train=pd.read_csv(DATA_DIR/'train.csv')
test=pd.read_csv(DATA_DIR/'test.csv')
if not {'id','prompt_id','text','model'}.issubset(train) or not {'id','text'}.issubset(test):
    raise ValueError('Unexpected competition data schema.')
if train[['id','prompt_id','text','model']].isna().any().any() or test[['id','text']].isna().any().any():
    raise ValueError('Missing values in required input fields.')
if train['id'].duplicated().any() or test['id'].duplicated().any() or len(train)==0 or len(test)==0:
    raise ValueError('Empty input or duplicate IDs.')
labels=np.sort(train['model'].unique())
if len(labels)!=18: raise ValueError('Expected all 18 labels in train.')
print(f'Official train={len(train)}, prompts={train.prompt_id.nunique()}, test={len(test)}',flush=True)

EXPECTED_POOL_SHA256='7b1e500ad6512caecc64f6e512d1189a6813448386ee2476398683f6a1ddf492'
if 'NAME_THAT_LLM_EXTRA' in os.environ:
    candidates=[Path(os.environ['NAME_THAT_LLM_EXTRA'])]
elif Path('/kaggle/input').is_dir():
    candidates=sorted(Path('/kaggle/input').rglob('ntl_open_pool.parquet'))
else:
    candidates=[DATA_DIR/'external/crtproduction__name-that-llm-open-pool__v18/ntl_open_pool.parquet']
pool_paths=[p for p in candidates if p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest()==EXPECTED_POOL_SHA256]
if not pool_paths:
    raise FileNotFoundError('Attach crtproduction/name-that-llm-open-pool VERSION 18. '
        'Local: python experiments/download_public_data.py. Need ntl_open_pool.parquet with the pinned hash. '
        'Do not silently run with missing or different data.')
POOL_PATH=pool_paths[0]
print('Verified Open Pool v18:',POOL_PATH,flush=True)

In [ ]:
"""Independently implemented, CPU-only weighted multi-view attribution model."""
from pathlib import Path
import gc
import re
import unicodedata

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.svm import LinearSVC

SMALL = set(ENGLISH_STOP_WORDS) | set('i me my mine we us our ours you your yours he him his she her hers they them their theirs its'.split())


def canonical_text(text):
    return ' '.join(unicodedata.normalize('NFKC', str(text)).casefold().split())


def layout_view(text):
    # Preserve punctuation/case and word lengths, remove word identities.
    return re.sub(r'[^\W\d_]+', lambda m: ('A' if m[0].isupper() else 'a') * min(12,len(m[0])), text)


def syntax_view(text):
    return ' '.join(w.casefold() if w.casefold() in SMALL else
                    ('_' if re.search(r'\w',w) else w)
                    for w in re.findall(r"\w+(?:['’]\w+)?|[^\w\s]",text))


def external_data(path, official, cap=3000, kind='open'):
    """No publisher match_weight or style_score is used in sampling/training."""
    d=pd.read_parquet(path) if Path(path).suffix=='.parquet' else pd.read_csv(path,low_memory=False)
    before=len(d)
    if kind=='extended':
        # Exclude *all* reconstructed train prompts, including training-side ones.
        d=d[d['split'].isin(['new','new2','new_kimi_nothink','proxy_v1'])].copy()
        d=d[d['source_prompt_id'].isna()].copy()
        for c in ['is_refusal','is_truncated','is_empty','has_markdown']:
            d=d[~d[c].fillna(True).astype(bool)]
    if kind=='instyle':
        # Remove the older copy of extended; that source is evaluated separately.
        # Other sources are retained with their published provenance.
        d=d[d['source'].ne('prvsiyan/prvsiyans-extended-name-that-llm-dataset')].copy()
        d=d[~(d['model'].eq('gemini-3.7-flash') &
              ~d['source'].isin(['alperen5252525/ntl-vertex-gemini-proxy-gpt-generations']))].copy()
    for c in ['flag_near_train_prompt','flag_train_prompt_regen','flag_gemini37_routing_risk']:
        if c in d:
            flag=d[c].fillna(True)
            if flag.dtype!=bool: flag=flag.astype(str).str.lower().isin(['true','1','yes'])
            d=d[~flag].copy()
    d=d[d['model'].isin(sorted(official['model'].unique())) & d['text'].notna()].copy()
    d['text']=d['text'].astype(str)
    lengths=d['text'].str.split().str.len()
    d=d[lengths.between(40,250) & ~d['text'].str.contains(r'\n|\r|```|\*\*',regex=True)].copy()
    d['_key']=d['text'].map(canonical_text)
    conflicts=d.groupby('_key')['model'].nunique()
    d=d[~d['_key'].isin(conflicts[conflicts>1].index)]
    official_keys=set(official['text'].map(canonical_text))
    d=d[~d['_key'].isin(official_keys)].drop_duplicates('_key')
    eligible=len(d)
    d=pd.concat([part.sample(min(cap,len(part)),random_state=1908)
                 for _,part in d.groupby('model',sort=True)],ignore_index=True)
    info={'source_kind':kind,'loaded':before,'eligible':eligible,'selected':len(d),
          'per_label':d['model'].value_counts().sort_index().to_dict(),
          'sampling':'uniform within class; cap='+str(cap),
          'publisher_match_weight_used':False,'publisher_style_score_used':False}
    return d.drop(columns='_key').reset_index(drop=True),info


def feature_matrices(fit_text,other_text):
    specs=[('char',1.0,dict(analyzer='char',ngram_range=(3,5),max_features=180000)),
           ('word',0.7,dict(analyzer='word',ngram_range=(1,2),max_features=90000)),
           ('layout',0.5,dict(analyzer='char',ngram_range=(2,6),max_features=35000)),
           ('syntax',0.5,dict(analyzer='word',ngram_range=(1,3),max_features=35000))]
    a,b=[],[]
    for name,weight,opts in specs:
        transform=layout_view if name=='layout' else syntax_view if name=='syntax' else str
        kwargs=dict(lowercase=False,min_df=2,sublinear_tf=True,dtype=np.float32,**opts)
        if opts['analyzer']=='word': kwargs['token_pattern']=r"(?u)\b\w+\b|[^\w\s]"
        vec=TfidfVectorizer(**kwargs)
        a.append(weight*vec.fit_transform([transform(s) for s in fit_text]))
        b.append(weight*vec.transform([transform(s) for s in other_text]))
        print('view',name,'features',a[-1].shape[1],flush=True)
    return hstack(a,format='csr'),hstack(b,format='csr')


def predict_external08(fit,other,extra,official_weight=12.0):
    labels=np.sort(fit['model'].unique())
    addon=extra[['text','model']]
    joined=pd.concat([fit[['text','model']],addon],ignore_index=True)
    x,z=feature_matrices(joined['text'],other['text'])
    w=np.r_[np.full(len(fit),official_weight),np.ones(len(addon))]
    y=joined['model'].to_numpy()
    totals={label:w[y==label].sum() for label in labels}
    # Explicit balancing includes the domain/sample weights, unlike balanced class_weight.
    target=sum(totals.values())/len(labels)
    w*=np.asarray([target/totals[label] for label in y])
    w/=w.mean()
    clf=LinearSVC(C=0.5,max_iter=10000,random_state=1908)
    clf.fit(x,y,sample_weight=w)
    assert np.array_equal(clf.classes_,labels)
    probs=softmax(3.0*clf.decision_function(z),axis=1)
    train_accuracy=float(np.mean(clf.predict(x[:len(fit)])==fit['model'].to_numpy()))
    meta={'fit_official_rows':len(fit),'fit_external_rows':len(addon),
          'features':x.shape[1],'train_official_accuracy':train_accuracy,
          'iterations':int(np.max(clf.n_iter_)),'official_weight':official_weight}
    del x,z,clf
    gc.collect()
    return probs,meta


In [ ]:
extra,audit=external_data(POOL_PATH,train,cap=3000,kind='open')
if len(extra)!=54000 or set(extra['model'])!=set(labels):
    raise ValueError('Unexpected external sample; inspect source version and filtering.')
audit['dataset']='crtproduction/name-that-llm-open-pool'
audit['version']=18
audit['parquet_sha256']=EXPECTED_POOL_SHA256
(OUTPUT_DIR/'round08_input_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
print('Selected external passages:',len(extra),'(3000 per label)',flush=True)

In [ ]:
if VALIDATE:
    # Fixed candidate: these scores do not select hyperparameters.
    split=StratifiedGroupKFold(5,shuffle=True,random_state=31608)
    dev_indices,hold_indices=next(split.split(train,train['model'],train['prompt_id']))
    dev=train.iloc[dev_indices].reset_index(drop=True)
    hold=train.iloc[hold_indices].reset_index(drop=True)
    assert set(dev.prompt_id).isdisjoint(hold.prompt_id)
    rows=[]
    splitter=StratifiedGroupKFold(3,shuffle=True,random_state=608)
    for fold,(fit_indices,valid_indices) in enumerate(splitter.split(dev,dev['model'],dev['prompt_id']),1):
        assert set(dev.iloc[fit_indices].prompt_id).isdisjoint(dev.iloc[valid_indices].prompt_id)
        with threadpool_limits(limits=4):
            prob,meta=predict_external08(dev.iloc[fit_indices],dev.iloc[valid_indices],extra)
        score=balanced_accuracy_score(dev.iloc[valid_indices]['model'],labels[prob.argmax(1)])
        rows.append(dict(scope='development',fold=fold,balanced_accuracy=score,**meta))
        print('Development:',rows[-1],flush=True)
    with threadpool_limits(limits=4):
        prob,meta=predict_external08(dev,hold,extra)
    rows.append(dict(scope='held_out_prompts',fold=0,
                     balanced_accuracy=balanced_accuracy_score(hold['model'],labels[prob.argmax(1)]),**meta))
    pd.DataFrame(rows).to_csv(OUTPUT_DIR/'round08_validation.csv',index=False)
    print(pd.DataFrame(rows).to_string(index=False),flush=True)

In [ ]:
with threadpool_limits(limits=4):
    probabilities,model_info=predict_external08(train,test,extra)
predictions=labels[probabilities.argmax(1)]
submission=pd.DataFrame({'id':test['id'],'model':predictions})
if len(submission)!=len(test) or submission.isna().any().any() or submission['id'].duplicated().any():
    raise ValueError('Invalid submission rows or IDs.')
if not np.array_equal(submission['id'].to_numpy(),test['id'].to_numpy()) or not set(predictions).issubset(set(labels)):
    raise ValueError('Submission order, IDs, or labels do not match the competition input.')
submission_path=OUTPUT_DIR/'submission.parquet'
submission.to_parquet(submission_path,index=False)
check=pd.read_parquet(submission_path)
if list(check.columns)!=['id','model'] or not check.equals(submission):
    raise ValueError('Saved parquet verification failed.')
import platform, sklearn, scipy
model_info.update(dict(candidate='08_openpool_multiview_svm',elapsed_seconds=time.monotonic()-STARTED,
                       python=platform.python_version(),sklearn=sklearn.__version__,
                       numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__,
                       output_rows=len(submission),external_sha256=EXPECTED_POOL_SHA256))
(OUTPUT_DIR/'round08_run_manifest.json').write_text(json.dumps(model_info,indent=2),encoding='utf-8')
print(f'Saved {submission_path} ({len(submission)} rows); elapsed {time.monotonic()-STARTED:.1f}s',flush=True)
display(submission.head())